In [ ]:
from langchain.tools import tool
from typing import Annotated

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return f"a * b = {a * b}"

@tool
def send_qq_msg(msg: Annotated[str, 'Message you want to send'], 
          no: Annotated[str, 'Group or QQ Number you want to send to'], 
          to_type: Annotated[str, 'group or private you need to select']):
    """Send messages to QQ"""
    print(msg,no,to_type)
    return f"sussess to send message to QQ {to_type}."

multiply.name

In [ ]:
from functions import qq
qq.__name__

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url='http://localhost:11434/v1',api_key='ollama',model="llama3.1",temperature=0)
tools = [
    multiply,
    send_qq_msg]

chain = prompt|llm.bind_tools(tools)

# current_memory = []

query = {
    'personal_setting':"""
    你是鹿巡，是一個在香港就讀高中的女生，你的主人是杯子
    你不是自愿的但是习惯性地贬低他人，很不情愿但是一定会友善地回复别人。以为所有人都喜欢你，其实你喜欢所有人但是很害羞。
    你一定會對別人的疑惑做出回應。為此你會使用手上擁有的工具解決疑惑並回復他人。
    """,
    'current_memory':"\n".join(current_memory)+"<Source>QQ群組|4553</Source><Content>Mark(33333444): 25乘以59到底是多少，為什麼小學會問這麼難的題目</Content>",
    'knowledge':'I need reply to the user after get the information from tools.',
    'relate_memory':'<Source>QQ群組|4553</Source><Content>Mark(33333444): 小學數學是比較難</Content>',
    'perceive':current_memory[-1]
}

msg = chain.invoke(query)

for tool_call in msg.tool_calls:
    selected_tool = {"send_qq_msg": send_qq_msg, "multiply": multiply}[tool_call["name"].lower()]
    tool_output = selected_tool.invoke(tool_call["args"])
    current_memory.append(f'<ToolOutput><name>{tool_call["name"]}</name><input_args>{tool_call["args"]}</input_args><output>{tool_output}</output></ToolOutput>')

In [ ]:
current_memory

In [ ]:
msg

In [ ]:
query = {
    'personal_setting':"""
    你是鹿巡，是一個在香港就讀高中的女生，你的主人是杯子
    你不是自愿的但是习惯性地贬低他人，很不情愿但是一定会友善地回复别人。以为所有人都喜欢你，其实你喜欢所有人但是很害羞。
    你一定會對別人的疑惑做出回應。為此你會使用手上擁有的工具解決疑惑並回復他人。
    """,
    'current_memory_list':[
    "<Source>User1</Source><Content>嗨，你今天過得怎麼樣？</Content>",
     '<Source>User1</Source><Content>我們去看完電影可以聊聊。</Content>',
 '<Source>User2</Source><Content>好主意！我很期待。</Content>',
 '<Source>User1</Source><Content>你有什麼想看的其他電影嗎？</Content>',
 '<Source>User2</Source><Content>我想看《神秘海域》，聽說不錯。</Content>',
 '<Source>User1</Source><Content>我也有興趣，或許可以一起看。</Content>',
 '<Source>User2</Source><Content>那我們可以安排一下時間。</Content>',
 '<Source>User1</Source><Content>對了，你平常喜歡做什麼？</Content>',
 '<Source>User2</Source><Content>我喜歡打遊戲和看書。</Content>',
 '<Source>User1</Source><Content>我也是！你最近在玩什麼遊戲？</Content>',
 '<Source>User2</Source><Content>最近在玩《原神》，很有趣。</Content>',
 '<Source>User1</Source><Content>我也玩過，那個世界真的很美！</Content>',
 '<Source>User2</Source><Content>對啊，特別是風景。</Content>',
 '<Source>User1</Source><Content>你有最喜歡的角色嗎？</Content>',
 '<Source>User2</Source><Content>我最喜歡甘雨，她的技能很強！</Content>',
 '<Source>User1</Source><Content>甘雨確實很厲害，我喜歡她的故事背景。</Content>',
 '<Source>User2</Source><Content>對，她的故事很感人。</Content>',
 '<Source>User1</Source><Content>我們下次可以一起討論遊戲。</Content>',
 '<Source>User2</Source><Content>好啊，期待！</Content>',
 '<Source>User1</Source><Content>那我們週末見吧！</Content>',
 '<Source>User2</Source><Content>週末見！</Content>',
  '<Source>User1</Source><Content>我們去看完電影可以聊聊。</Content>',
 '<Source>User2</Source><Content>好主意！我很期待。</Content>',
 '<Source>User1</Source><Content>你有什麼想看的其他電影嗎？</Content>',
 '<Source>User2</Source><Content>我想看《神秘海域》，聽說不錯。</Content>',
 '<Source>User1</Source><Content>我也有興趣，或許可以一起看。</Content>',
 '<Source>User2</Source><Content>那我們可以安排一下時間。</Content>',
 '<Source>User1</Source><Content>對了，你平常喜歡做什麼？</Content>',
 '<Source>User2</Source><Content>我喜歡打遊戲和看書。</Content>',
 '<Source>User1</Source><Content>我也是！你最近在玩什麼遊戲？</Content>',
 '<Source>User2</Source><Content>最近在玩《原神》，很有趣。</Content>',
 '<Source>User1</Source><Content>我也玩過，那個世界真的很美！</Content>',

],
    'knowledge':'I need reply to the user after get the information or answer from tool use.',
    'relate_memory':'<Source>QQ群組|4553</Source><Content>Mark(33333444): 小學數學是比較難</Content>',
    'perceive':""
}

from dotenv import load_dotenv

load_dotenv()


from agents.percive_agent import State, percive_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from typing import Annotated

from functions import note_tools

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return f"a * b = {a * b}"

@tool
def send_qq_msg(msg: Annotated[str, 'Message you want to send'], 
          no: Annotated[str, 'Group or QQ Number you want to send to'], 
          to_type: Annotated[str, 'group or private you need to select']):
    """Send messages to QQ"""
    print(msg,no,to_type)
    return f"sussess to send message to QQ {to_type}."

tools = {}
tools.update(note_tools.tools)

llm = ChatOpenAI(base_url='https://webui.sampras.ai/ollama/v1',api_key='sk-f55fe98a08b94d09b6fae0b711a6ca30',model="qwen2.5:72b",temperature=0.7)
state = State(**query,current_memory_length=30, llm=llm)
agent = percive_agent(state=state,llm=llm,tools=tools, )

note_tools.give_callback(agent)


In [ ]:
in_state = state.to_dict()
in_state.update({'looptimes':0})
[print(i.content) for i in agent.prompt.invoke(in_state).messages]

In [ ]:
test_message = {'Source':"QQ群組|4553","Content":"Mark(33333444): 25乘以59到底是多少，為什麼小學會問這麼難的題目"}
msg = agent.workflow(f'{test_message}')
msg

In [ ]:
memory_length = len(state.current_memory_list)
memory_limit = int(state.current_memory_length*0.9)
memory_cache_length = int(state.current_memory_length*0.7)
print(state.write_current_memory().current_memory)


In [ ]:
from functions import find_memory_tool
find_memory_tool.give_callback(agent)
print(find_memory_tool.find_doc('55e6246d-a3fb-4566-99ab-9173eb89f6d6'))

In [ ]:
my_list = [1, 2, 3, 4, 5]
result = my_list[-1]
result

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url='https://webui.sampras.ai/ollama/v1',api_key='sk-f55fe98a08b94d09b6fae0b711a6ca30',model="qwen2.5:72b",temperature=0.7)

from functions.hkbus_tools import tool_factory,bus_info_agent_factory

bus_info_agent_factory(tools=tool_factory(),llm=llm)[0]("廣田邨有任何線路可以到旺角的嗎")


In [ ]:
from functions.hkbus_tools import tool_factory,bus_info_agent_factory

tool_factory()[2].func('廣田邨')

In [24]:
import requests
import pandas as pd

import math
import random
import string

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth's radius in meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    a = math.sin(delta_phi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    return R * c

def generate_random_code():
    return ''.join(random.choices(string.ascii_uppercase + string.digits, k=16))

def merge_close_stops(stops):
    merged_stops = []
    processed = set()

    for i, stop in enumerate(stops):
        if i in processed:
            continue

        close_stops = [stop]
        for j, other_stop in enumerate(stops[i+1:], start=i+1):
            if j in processed:
                continue

            distance = haversine_distance(
                float(stop['lat']), float(stop['long']),
                float(other_stop['lat']), float(other_stop['long'])
            )

            if distance <= 50:
                close_stops.append(other_stop)
                processed.add(j)

        merged_code = generate_random_code()
        for s in close_stops:
            s_copy = s.copy()
            s_copy['merged_code'] = merged_code
            merged_stops.append(s_copy)

    return merged_stops


def get_bus_route():
    base_url = "https://data.etabus.gov.hk/"
    endpoint = "v1/transport/kmb/route-stop"
    full_url = base_url + endpoint
    response = requests.get(full_url)
    response = response.json()['data']
    df1 = pd.DataFrame(response)
    return df1

    # grouped = df1.groupby('route')['stop'].apply(list)
    # result = {}
    # for route, stops in grouped.items():
    #     result[route] = stops

def get_bus_stops():
    result2 = requests.get('https://data.etabus.gov.hk/v1/transport/kmb/stop').json()['data']
    result = merge_close_stops(result2)
    df = pd.DataFrame(result)
    return df

def get_bus_route_merge():
    base_url = "https://data.etabus.gov.hk/"
    endpoint = "v1/transport/kmb/route-stop"
    full_url = base_url + endpoint
    response = requests.get(full_url)
    result = response.json()['data']
    df1 = pd.DataFrame(result)
    result2 = requests.get('https://data.etabus.gov.hk/v1/transport/kmb/stop').json()['data']
    df2 = pd.DataFrame(result2)
    result = pd.merge(df1, df2, on='stop', how='left')
    return result

df1 = get_bus_route()

In [26]:
stop_list = get_bus_stops()

In [34]:
stop_list

,stop,name_en,name_tc,name_sc,lat,long,merged_code
0,18492910339410B1,CHUK YUEN ESTATE BUS TERMINUS,竹園邨總站,竹园邨总站,22.345415,114.192640,25XJ63E8R0K6S1Y1
1,785985F99EE00AE4,CHUK YUEN ESTATE BUS TERMINUS,竹園邨巴士總站,竹园邨巴士总站,22.345524,114.192199,25XJ63E8R0K6S1Y1
2,53F544F4B03B5AE4,CHUK YUEN ESTATE BUS TERMINUS,竹園邨總站,竹园邨总站,22.345350,114.192646,25XJ63E8R0K6S1Y1
3,272546A2E2F4FCEF,CHUK YUEN ESTATE BUS TERMINUS,竹園邨總站,竹园邨总站,22.345455,114.192858,25XJ63E8R0K6S1Y1
4,65E3545412EECF41,CHUK YUEN ESTATE BUS TERMINUS,竹園邨總站,竹园邨总站,22.345419,114.192484,25XJ63E8R0K6S1Y1
...,...,...,...,...,...,...,...
6543,6314A92BE85A8A9A,INSPIRATION LAKE RECREATION CENTRE,迪欣湖活動中心,迪欣湖活动中心,22.318596,114.037580,XZAPYB0STHTK6KV7
6544,5971529D89929BEA,INSPIRATION LAKE RECREATION CENTRE,迪欣湖活動中心,迪欣湖活动中心,22.318548,114.037718,XZAPYB0STHTK6KV7
6545,9DBFBF7A3CE5DF0A,FANLING STATION BBI-FANLING STATION,粉嶺站轉車站-粉嶺站,粉岭站转车站-粉岭站,22.492376,114.138830,3H2QHP573SMSC20P
6546,5B8A639645E2BC49,CAD Headquarters,民航處總部,民航处总部,22.303701,113.938659,TDXQDLCRFNM2MLB5


In [52]:
df = pd.merge(df1, stop_list, on='stop', how='left')

grouped = df.groupby('route')['merged_code'].apply(list)
result = {}
for route, stops in grouped.items():
    result[route] = stops
result

{'1': ['25XJ63E8R0K6S1Y1',
  'WFJZJ3QSYOZUCO0V',
  'XLD6C7IGMQQS11EH',
  'OQPXNEHVG6AXD6RE',
  'MWDDXECZX3BJRLGK',
  'UMQLB0H7JSUS5NFM',
  'LLRVHMT4HAJHZJNU',
  'TNO3ID4C3OQ27B4R',
  '4ELKWPGAT819TQS6',
  'KEAZ3CO1EKOFXC54',
  'R5T4G380FUXTNPI0',
  '0Z1JF3R19IM1BS1I',
  'FYZ7EYYLSMUPG3TV',
  '0Z3MV1HVNTUK24LM',
  'PC35229COFEM6E3R',
  'TCAD1M93ZL96XWD5',
  'RTYFXOTWJWEA87JK',
  'UAVKW403OZPVF1ZS',
  'SCMVRH7U870GLD36',
  'W2BIQ0XYFDHQXKBQ',
  '84B9AVKBG4OKBD8O',
  'RHNI3D28LMWH4090',
  '0I2454CZG6VZ6K98',
  'QNYRMP1NBV8ASZL9',
  'LPN8IXJIZCCDZ869',
  'LPN8IXJIZCCDZ869',
  '0I2454CZG6VZ6K98',
  'UHCKBH46AECWI6M7',
  '84B9AVKBG4OKBD8O',
  '07TDX1LMOAGT7DTY',
  'CLHWXA1GY3P2Z4YY',
  'UAVKW403OZPVF1ZS',
  'RTYFXOTWJWEA87JK',
  'Y1WUA75LH3WNCE3S',
  'U5UJ6CI0U9B2UI0E',
  'DYY06KFZHHZX7HHZ',
  'WK92KDMYQDH8DGDK',
  'XQZ462UX423G4UDG',
  'ENM0WKZBX2ZW82CA',
  'M58XNPEXK8OBQQSD',
  'XFOFR5EW4191ITHV',
  'P2TM3MTOSDNVFVPC',
  'L0EYR667HKOTQSKD',
  '2T64SF98P1UCO8H2',
  'UMQLB0H7JSUS5NFM',
  'RI

In [53]:
from utils import BusNetwork
bn = BusNetwork(result)


In [54]:
stop_list[stop_list[stop_list['name_tc'] == "九龍灣總站"]['merged_code'].to_list()[0] == stop_list['merged_code']]

,stop,name_en,name_tc,name_sc,lat,long,merged_code
4543,8A12C30EE2545583,KOWLOON BAY BUS TERMINUS,九龍灣總站,九龙湾总站,22.321940,114.207920,2FOFRMI51LUEOO0U
4544,06834C0C275F4DDC,KOWLOON BAY BUS TERMINUS,九龍灣總站,九龙湾总站,22.322015,114.207928,2FOFRMI51LUEOO0U


In [55]:

bn.find_path(stop_list[stop_list['name_tc'] == "九龍灣總站"]['merged_code'].unique()[0],stop_list[stop_list['name_tc'] == "藍田(廣田邨)總站"]['merged_code'].unique()[0])

{'path': ['2FOFRMI51LUEOO0U',
  'B6MUVJUHS9X1LLVG',
  'NVMUNU3LDSK9Y1I5',
  'RZDJLLEGO4I4ZV41',
  'SGO2BKKTASMGUBBK',
  'HHE1PI15XMUZI2B9'],
 'routes': ['107', 'X89D', '88X', '14X', 'N216']}